# Ground-truth simulation benchmark - FastICA split

This notebook is a method-specific split of `simulation_benchmark_all_methods.ipynb`. It runs only `fastica` BSS methods from `configs/simulation.core.json` and writes to `outputs/simulation/core_1000_fastica/`, so it can run in parallel with the other split notebooks without sharing replicate manifests.


In [1]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = (start, *start.parents)
    for path in candidates:
        if (path / "pyproject.toml").exists() and (path / "src" / "ica_denoising").exists():
            return path
    raise RuntimeError("Could not find the ica-denoising repository root.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))
(REPO_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def refresh_simulation_modules() -> None:
    # Notebook kernels keep old project modules after source edits; reload this package family.
    for module_name in tuple(sys.modules):
        if (
            module_name == "ica_denoising.simulation"
            or module_name.startswith("ica_denoising.simulation.")
            or module_name == "csl.experiments.simulation_adapters"
        ):
            del sys.modules[module_name]


if "ipykernel" in sys.modules:
    refresh_simulation_modules()


def repo_display_path(path: str | Path) -> str:
    path = Path(path)
    if not path.is_absolute():
        return str(path)
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)

import matplotlib
if "ipykernel" in sys.modules:
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("matplotlib", "inline")
    except Exception:
        pass
else:
    matplotlib.use("Agg")
SHOW_FIGURES = "ipykernel" in sys.modules

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ica_denoising.simulation import (
    import_completed_replicates_from_benchmark,
    load_config,
    run_benchmark,
)
from ica_denoising.simulation.reporting import (
    collect_metric_table,
    paired_vs_raw,
    summarize_graph_recovery,
)
from ica_denoising.simulation.validate import validate_benchmark

sns.set_theme(style="whitegrid", context="talk")
print(f"Repository root: {REPO_ROOT}")

Repository root: /Users/sadiqadedayo/Documents/projects/ICA/ica-denoising


## Run configuration

In [2]:

# Rebind simulation imports in case this cell is rerun after a source/schema edit.
def refresh_simulation_modules() -> None:
    for module_name in tuple(sys.modules):
        if (
            module_name == "ica_denoising.simulation"
            or module_name.startswith("ica_denoising.simulation.")
            or module_name == "csl.experiments.simulation_adapters"
        ):
            del sys.modules[module_name]


if "ipykernel" in sys.modules:
    refresh_simulation_modules()
from ica_denoising.simulation import (
    import_completed_replicates_from_benchmark,
    load_config,
    run_benchmark,
)
from ica_denoising.simulation.reporting import (
    collect_metric_table,
    paired_vs_raw,
    summarize_graph_recovery,
)
from ica_denoising.simulation.validate import validate_benchmark

CONFIG_PATH = REPO_ROOT / "configs" / "simulation.core.json"
METHOD_SPLIT_NAME = 'fastica'
METHODS_TO_RUN = ('fastica',)

# Keep these as None to use the selected config. For a quick single replicate,
# set SCENARIOS = ["S0"] and SEEDS = [0].
SCENARIOS: list[str] | None = None
SEEDS: list[int] | None = None

RUN_BENCHMARK = True
DRY_RUN = False
RESUME = True

MAIN_GRAPH_ESTIMATORS = ("cgc", "cgc_star", "pcmci", "jpcmciplus")
SUPPLEMENTARY_GRAPH_ESTIMATORS = ("var",)
IMPORT_COMPLETED_FROM_UNSPLIT = False
UNSPLIT_BENCHMARK_VERSION = "core_1000"

cfg = load_config(CONFIG_PATH)
cfg = replace(
    cfg,
    benchmark_version=f"{cfg.benchmark_version}_{METHOD_SPLIT_NAME}",
    bss=replace(cfg.bss, methods=METHODS_TO_RUN),
)
benchmark_root = REPO_ROOT / cfg.run.output_dir / cfg.benchmark_version
unsplit_benchmark_root = REPO_ROOT / cfg.run.output_dir / UNSPLIT_BENCHMARK_VERSION
summary_dir = benchmark_root / "summary"
figure_dir = benchmark_root / "figures"

print(json.dumps({
    "config": str(CONFIG_PATH.relative_to(REPO_ROOT)),
    "method_split": METHOD_SPLIT_NAME,
    "methods": list(cfg.bss.methods),
    "benchmark_version": cfg.benchmark_version,
    "benchmark_root": str(benchmark_root.relative_to(REPO_ROOT)),
    "import_source": str(unsplit_benchmark_root.relative_to(REPO_ROOT)),
    "import_completed_from_unsplit": IMPORT_COMPLETED_FROM_UNSPLIT,
    "scenarios": SCENARIOS or "from config",
    "seeds": SEEDS or "from config",
    "dry_run": DRY_RUN,
    "resume": RESUME,
    "main_graph_estimators": list(MAIN_GRAPH_ESTIMATORS),
    "supplementary_graph_estimators": list(SUPPLEMENTARY_GRAPH_ESTIMATORS),
}, indent=2))


{
  "config": "configs/simulation.core.json",
  "method_split": "fastica",
  "methods": [
    "fastica"
  ],
  "benchmark_version": "core_1000_fastica",
  "benchmark_root": "outputs/simulation/core_1000_fastica",
  "import_source": "outputs/simulation/core_1000",
  "import_completed_from_unsplit": false,
  "scenarios": "from config",
  "seeds": "from config",
  "dry_run": false,
  "resume": true,
  "main_graph_estimators": [
    "cgc",
    "cgc_star",
    "pcmci",
    "jpcmciplus"
  ],
  "supplementary_graph_estimators": [
    "var"
  ]
}


## Execute benchmark

In [3]:
def format_progress_bar(completed, total, width=28):
    total = max(int(total), 1)
    completed = min(max(int(completed), 0), total)
    filled = int(round(width * completed / total))
    return f"[{'#' * filled}{'.' * (width - filled)}] {completed}/{total}"


def make_progress_callback(label):
    last = {"line": ""}

    def callback(event):
        status = str(event.get("event", ""))
        scenario = event.get("scenario_id", "?")
        seed = event.get("seed", "?")
        completed = int(event.get("completed", 0))
        total = int(event.get("total", 0))
        line = f"{label}: {format_progress_bar(completed, total)} {status} {scenario}/seed_{seed}"
        padding = " " * max(0, len(last["line"]) - len(line))
        print("\r" + line + padding, end="", flush=True)
        last["line"] = line
        if status not in {"start"}:
            print("", flush=True)
            last["line"] = ""

    return callback


if IMPORT_COMPLETED_FROM_UNSPLIT and not DRY_RUN:
    import_report = import_completed_replicates_from_benchmark(
        unsplit_benchmark_root,
        cfg,
        scenarios=SCENARIOS,
        seeds=SEEDS,
        resume=RESUME,
        progress_callback=make_progress_callback("import"),
    )
else:
    import_report = {}

if import_report:
    print("Import summary:", json.dumps(import_report, indent=2))

if RUN_BENCHMARK:
    planned_paths = run_benchmark(
        cfg,
        scenarios=SCENARIOS,
        seeds=SEEDS,
        resume=RESUME,
        dry_run=DRY_RUN,
        progress_callback=make_progress_callback("run"),
    )
else:
    planned_paths = sorted(benchmark_root.glob("*/seed_*"))

print(f"{'Planned' if DRY_RUN else 'Available'} replicate(s): {len(planned_paths)}")
for path in planned_paths:
    print(repo_display_path(path))


run: [............................] 1/240 skipped S0/seed_0
run: [............................] 2/240 skipped S0/seed_1
run: [............................] 3/240 skipped S0/seed_2
run: [............................] 4/240 skipped S0/seed_3
run: [#...........................] 5/240 skipped S0/seed_4
run: [#...........................] 6/240 skipped S0/seed_5
run: [#...........................] 7/240 skipped S0/seed_6
run: [#...........................] 8/240 skipped S0/seed_7
run: [#...........................] 9/240 skipped S0/seed_8
run: [#...........................] 10/240 skipped S0/seed_9
run: [#...........................] 11/240 skipped S0/seed_10
run: [#...........................] 12/240 skipped S0/seed_11
run: [##..........................] 13/240 skipped S0/seed_12
run: [##..........................] 14/240 skipped S0/seed_13
run: [##..........................] 15/240 skipped S0/seed_14
run: [##..........................] 16/240 skipped S0/seed_15
run: [##...................

/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the max

run: [##############..............] 117/240 done S3/seed_26 
run: [##############..............] 117/240 start S3/seed_27

/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the max

run: [##############..............] 118/240 done S3/seed_27 
run: [##############..............] 118/240 start S3/seed_28

...

run: [#################...........] 144/240 done S4/seed_23 
run: [#################...........] 144/240 start S4/seed_24

/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the max

run: [#################...........] 145/240 done S4/seed_24 
run: [#################...........] 145/240 start S4/seed_25

/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/Users/sadiqadedayo/Documents/projects/ICA/ica-denoising/.venv/lib/python3.13/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the max

KeyboardInterrupt: 

## Manifest validation

In [ ]:
if DRY_RUN:
    print("Dry run selected; no manifests were written.")
else:
    report = validate_benchmark(benchmark_root)
    print(f"Checked {report.checked} replicate(s).")
    if not report.ok:
        for error in report.errors:
            print(f"- {error}")
        raise RuntimeError("Simulation benchmark validation failed.")
    print("OK: manifest and output validation passed.")

## Replicate manifest overview

In [ ]:
manifest_rows = []
for manifest_path in sorted(benchmark_root.glob("*/seed_*/manifest.json")):
    manifest = json.loads(manifest_path.read_text())
    manifest_rows.append({
        "scenario": manifest.get("scenario_id"),
        "seed": manifest.get("seed"),
        "complete": manifest.get("complete"),
        "failures": len(manifest.get("failures", [])),
        "n_variants": manifest.get("n_variants"),
        "achieved_asr": manifest.get("achieved_asr"),
        "runtime_seconds": manifest.get("runtime_seconds"),
        "git_revision": manifest.get("git_revision"),
    })

manifest_table = pd.DataFrame(manifest_rows)
manifest_table

## Rebuild summary tables

In [ ]:
summary_dir.mkdir(parents=True, exist_ok=True)


def keep_main_graph_estimators(table: pd.DataFrame) -> pd.DataFrame:
    if table.empty or "estimator" not in table.columns:
        return table.copy()
    return table[table["estimator"].isin(MAIN_GRAPH_ESTIMATORS)].copy()


metric_tables = {}
for family in ("trace", "behavior", "state", "graph"):
    table = collect_metric_table(benchmark_root, family)
    metric_tables[family] = table
    table.to_csv(summary_dir / f"{family}_all.csv", index=False)
    print(f"{family}: {len(table)} row(s)")

graph_summary_all = summarize_graph_recovery(benchmark_root)
graph_paired_f1_all = paired_vs_raw(benchmark_root, metric="f1")
graph_summary = keep_main_graph_estimators(graph_summary_all)
graph_paired_f1 = keep_main_graph_estimators(graph_paired_f1_all)

graph_summary_all.to_csv(
    summary_dir / "graph_recovery_summary_all_estimators.csv", index=False
)
graph_paired_f1_all.to_csv(
    summary_dir / "graph_paired_vs_raw_all_estimators.csv", index=False
)
graph_summary.to_csv(summary_dir / "graph_recovery_summary.csv", index=False)
graph_paired_f1.to_csv(summary_dir / "graph_paired_vs_raw.csv", index=False)

if "estimator" in graph_summary_all.columns:
    all_estimators = set(graph_summary_all["estimator"].dropna().astype(str))
    main_estimators = set(graph_summary["estimator"].dropna().astype(str))
    excluded = sorted(all_estimators - main_estimators)
    if excluded:
        print(
            "Supplementary/internal estimator rows excluded from main summaries:",
            excluded,
        )

print(f"Wrote summaries to {summary_dir.relative_to(REPO_ROOT)}")
graph_summary.head(20)


## Primary graph-recovery comparison

In [ ]:
graph_all = metric_tables["graph"]
graph = keep_main_graph_estimators(graph_all)
if graph_all.empty:
    raise RuntimeError("No graph metrics found. Run the benchmark first.")
if graph.empty:
    raise RuntimeError(
        "No main graph metrics found for MAIN_GRAPH_ESTIMATORS. "
        "Check estimator outputs or adjust the manuscript-facing estimator filter."
    )

primary_columns = [
    "scenario",
    "estimator",
    "variant_id",
    "f1",
    "precision",
    "recall",
    "shd",
    "edge_density_bias",
    "jaccard_vs_reference",
]
available = [col for col in primary_columns if col in graph.columns]
(
    graph[available]
    .sort_values(["scenario", "estimator", "f1"], ascending=[True, True, False])
    .head(30)
)

In [ ]:
figure_dir.mkdir(parents=True, exist_ok=True)

plot_data = graph_paired_f1[graph_paired_f1["variant_id"] != "raw"].copy()
if plot_data.empty:
    print("No paired graph data to plot.")
else:
    height = max(5, 0.35 * plot_data["variant_id"].nunique())
    g = sns.catplot(
        data=plot_data,
        kind="bar",
        x="f1_minus_raw",
        y="variant_id",
        hue="estimator",
        col="scenario",
        col_wrap=2,
        height=height,
        aspect=1.25,
        errorbar=None,
        sharex=True,
        sharey=False,
    )
    g.set_axis_labels("F1 minus raw", "Variant")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.axvline(0.0, color="black", linewidth=1)
    g.fig.suptitle("Paired graph-recovery effect relative to raw", y=1.02)
    out = figure_dir / "graph_f1_minus_raw.png"
    g.fig.savefig(out, dpi=200, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(g.fig)
    print(f"Saved {out.relative_to(REPO_ROOT)}")

## Trace, behavior, and state trade-off table

In [ ]:
trace = metric_tables["trace"]
behavior = metric_tables["behavior"]
state = metric_tables["state"]

trace_med = trace.groupby(["scenario", "variant_id"], as_index=False).agg(
    trace_pearson_global=("trace_pearson_global", "median"),
    trace_nrmse=("trace_nrmse", "median"),
    artifact_residual_corr=("artifact_residual_corr", "median"),
    effective_rank=("effective_rank", "median"),
)
behavior_med = behavior.groupby(["scenario", "variant_id"], as_index=False).agg(
    behavior_pearson=("behavior_pearson", "median"),
    bout_balanced_accuracy=("bout_balanced_accuracy", "median"),
)
state_med = state.groupby(["scenario", "variant_id"], as_index=False).agg(
    persistence_improvement=("persistence_improvement", "median"),
    forward_reverse_gap=("forward_reverse_gap", "median"),
    residual_artifact_assoc=("residual_artifact_assoc", "median"),
)
graph_med = graph.groupby(["scenario", "variant_id"], as_index=False).agg(
    graph_f1=("f1", "median"),
    graph_shd=("shd", "median"),
)

tradeoff = trace_med.merge(behavior_med, on=["scenario", "variant_id"], how="outer")
tradeoff = tradeoff.merge(state_med, on=["scenario", "variant_id"], how="outer")
tradeoff = tradeoff.merge(graph_med, on=["scenario", "variant_id"], how="outer")
tradeoff.to_csv(summary_dir / "trace_behavior_state_graph_tradeoff.csv", index=False)
tradeoff.sort_values(["scenario", "graph_f1", "trace_pearson_global"], ascending=[True, False, False]).head(30)

In [ ]:
if tradeoff.empty:
    print("No trade-off data to plot.")
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        data=tradeoff,
        x="trace_pearson_global",
        y="graph_f1",
        hue="scenario",
        style="variant_id",
        s=90,
        ax=ax,
    )
    ax.set_xlabel("Trace recovery, median Pearson vs clean")
    ax.set_ylabel("Graph recovery, median F1")
    ax.set_title("Trace recovery versus graph recovery")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    out = figure_dir / "trace_vs_graph_recovery.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)
    print(f"Saved {out.relative_to(REPO_ROOT)}")

## Benchmark audit flags

In [ ]:
audit_flags = []
if cfg.estimator.correction == "fdr":
    audit_flags.append({
        "level": "review",
        "item": "FDR graph scoring",
        "note": "The c-GC adapter computes binary_fdr, but current runner outputs should be checked before treating FDR-corrected graphs as the scored primary graph.",
    })
if len(cfg.bss.random_states) > 1 or len(cfg.bss.ranks) > 1:
    audit_flags.append({
        "level": "review",
        "item": "BSS grid expansion",
        "note": "The config can list multiple ranks/seeds; verify the runner expands them before using core or robustness as final scientific runs.",
    })
if manifest_table.get("failures", pd.Series(dtype=int)).sum() > 0:
    audit_flags.append({
        "level": "blocker",
        "item": "Replicate failures",
        "note": "At least one manifest recorded failures. Inspect manifest['failures'] before using summaries.",
    })

pd.DataFrame(audit_flags)